# Homework 3. SQL, Data Loading & EDA
## Task 1. Setup and data loading

Для цього завдання я обрала датасет Adult з UCI Machine Learning Repository. Він містить інформацію про людей, зокрема їхній вік, освіту, професію, кількість робочих годин та рівень доходу.

Мета цього датасету - визначити, чи перевищує річний дохід людини 50 000 доларів.

Я завантажила файл `adult.data` з офіційного джерела. Він містить 32 561 запис.

Спочатку створила таблицю `hw3_adult_staging`, у якій всі колонки мають тип `TEXT`. Це потрібно, щоб завантажити дані без попереднього очищення, включно зі спеціальними значеннями `?`, які позначають пропуски.

Для завантаження використала команду PostgreSQL `COPY FROM STDIN`. Після цього перевірила кількість рядків і переглянула перші записи.

In [1]:
!pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

# Start PostgreSQL 16
pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

# Check PostgreSQL version
with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print(version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 62.8 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
from urllib.request import urlretrieve

# Download the original Adult dataset from UCI
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

csv_path = "/content/hw3_adult.csv"
urlretrieve(DATA_URL, csv_path)

# Count rows without changing the raw data
with open(csv_path, "r", encoding="utf-8") as file:
    row_count = sum(1 for line in file if line.strip())

print(f"Downloaded rows: {row_count}")
print(f"CSV path: {csv_path}")

Downloaded rows: 32561
CSV path: /content/hw3_adult.csv


In [3]:
# Create a staging table for raw CSV values
staging_sql = """
DROP TABLE IF EXISTS hw3_adult_clean CASCADE;
DROP TABLE IF EXISTS hw3_adult_staging CASCADE;

CREATE TABLE hw3_adult_staging (
    age_raw TEXT,
    workclass_raw TEXT,
    fnlwgt_raw TEXT,
    education_raw TEXT,
    education_num_raw TEXT,
    marital_status_raw TEXT,
    occupation_raw TEXT,
    relationship_raw TEXT,
    race_raw TEXT,
    sex_raw TEXT,
    capital_gain_raw TEXT,
    capital_loss_raw TEXT,
    hours_per_week_raw TEXT,
    native_country_raw TEXT,
    income_raw TEXT
);
"""

with engine.begin() as conn:
    conn.execute(text(staging_sql))

print("Staging table created successfully.")

Staging table created successfully.


In [5]:
import io

# Read the CSV and remove empty lines
with open(csv_path, "r", encoding="utf-8") as file:
    csv_content = "".join(
        line for line in file if line.strip()
    )

# Load the CSV using COPY FROM STDIN
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        cur.copy_expert("""
            COPY hw3_adult_staging (
                age_raw,
                workclass_raw,
                fnlwgt_raw,
                education_raw,
                education_num_raw,
                marital_status_raw,
                occupation_raw,
                relationship_raw,
                race_raw,
                sex_raw,
                capital_gain_raw,
                capital_loss_raw,
                hours_per_week_raw,
                native_country_raw,
                income_raw
            )
            FROM STDIN
            WITH (
                FORMAT csv,
                HEADER false,
                DELIMITER ','
            )
        """, io.StringIO(csv_content))

    raw_conn.commit()
finally:
    raw_conn.close()

print("CSV imported successfully.")

CSV imported successfully.


In [6]:
# Verify row count
df = pd.read_sql(text("""
    SELECT COUNT(*) AS loaded_rows
    FROM hw3_adult_staging
"""), engine)

display(df)

# Preview raw records
df = pd.read_sql(text("""
    SELECT
        age_raw,
        workclass_raw,
        education_raw,
        occupation_raw,
        hours_per_week_raw,
        income_raw
    FROM hw3_adult_staging
    LIMIT 10
"""), engine)

display(df)

,loaded_rows
0,32561


,age_raw,workclass_raw,education_raw,occupation_raw,hours_per_week_raw,income_raw
0,39,State-gov,Bachelors,Adm-clerical,40,<=50K
1,50,Self-emp-not-inc,Bachelors,Exec-managerial,13,<=50K
2,38,Private,HS-grad,Handlers-cleaners,40,<=50K
3,53,Private,11th,Handlers-cleaners,40,<=50K
4,28,Private,Bachelors,Prof-specialty,40,<=50K
5,37,Private,Masters,Exec-managerial,40,<=50K
6,49,Private,9th,Other-service,16,<=50K
7,52,Self-emp-not-inc,HS-grad,Exec-managerial,45,>50K
8,31,Private,Masters,Prof-specialty,50,>50K
9,42,Private,Bachelors,Exec-managerial,40,>50K


## 2. Staging та очищення даних

Спочатку всі дані були завантажені в таблицю `hw3_adult_staging` як `TEXT`. Тепер потрібно створити окрему таблицю з правильними типами даних.

Для числових колонок я використала `INTEGER`, для текстових - `TEXT`, а для цільової змінної `income` - `BOOLEAN`, оскільки вона має тільки два можливі значення.

У вихідному файлі всі значення записані як текст. У деяких категоріальних колонках є символ `?`, який означає пропущене значення. Під час очищення такі значення перетворюються на SQL `NULL`, а зайві пробіли видаляються.

| Колонка | Що означає | Тип | Очищення |
|---|---|---|---|
| `age` | Вік людини | INTEGER | Перетворення на число |
| `workclass` | Тип зайнятості | TEXT | Видалення пробілів, `?` → NULL |
| `fnlwgt` | Статистична вага запису | INTEGER | Перетворення на число |
| `education` | Рівень освіти | TEXT | Видалення пробілів |
| `education_num` | Числовий код рівня освіти | INTEGER | Перетворення на число |
| `marital_status` | Сімейний стан | TEXT | Видалення пробілів |
| `occupation` | Професія | TEXT | Видалення пробілів, `?` → NULL |
| `relationship` | Сімейні відносини | TEXT | Видалення пробілів |
| `race` | Категорія, вказана в датасеті | TEXT | Видалення пробілів |
| `sex` | Стать у датасеті | TEXT | Видалення пробілів |
| `capital_gain` | Дохід від капіталу | INTEGER | Перетворення на число |
| `capital_loss` | Втрати капіталу | INTEGER | Перетворення на число |
| `hours_per_week` | Робочі години на тиждень | INTEGER | Перетворення на число |
| `native_country` | Країна походження | TEXT | Видалення пробілів, `?` → NULL |
| `income` | Чи перевищує дохід 50 000 доларів | BOOLEAN | `>50K` → TRUE, `<=50K` → FALSE |

Окремий `record_id` буде технічним первинним ключем. Він потрібен, щоб кожен рядок мав унікальний ідентифікатор.

У цьому датасеті немає колонок із календарними датами або точними часовими мітками, тому типи `DATE` і `TIMESTAMPTZ` тут не потрібні.

In [7]:
# Create the final typed table
clean_table_sql = """
DROP TABLE IF EXISTS hw3_adult_clean CASCADE;

CREATE TABLE hw3_adult_clean (
    record_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    age INTEGER NOT NULL CHECK (age BETWEEN 0 AND 120),
    workclass TEXT,
    fnlwgt INTEGER NOT NULL CHECK (fnlwgt > 0),
    education TEXT NOT NULL,
    education_num INTEGER NOT NULL
        CHECK (education_num BETWEEN 1 AND 16),
    marital_status TEXT NOT NULL,
    occupation TEXT,
    relationship TEXT NOT NULL,
    race TEXT NOT NULL,
    sex TEXT NOT NULL,
    capital_gain INTEGER NOT NULL CHECK (capital_gain >= 0),
    capital_loss INTEGER NOT NULL CHECK (capital_loss >= 0),
    hours_per_week INTEGER NOT NULL
        CHECK (hours_per_week BETWEEN 0 AND 168),
    native_country TEXT,
    income BOOLEAN NOT NULL
);
"""

with engine.begin() as conn:
    conn.execute(text(clean_table_sql))

print("Clean table created successfully.")

Clean table created successfully.


In [8]:
# Clean staging data and insert it into the typed table
insert_clean_sql = """
INSERT INTO hw3_adult_clean (
    age,
    workclass,
    fnlwgt,
    education,
    education_num,
    marital_status,
    occupation,
    relationship,
    race,
    sex,
    capital_gain,
    capital_loss,
    hours_per_week,
    native_country,
    income
)
SELECT
    TRIM(age_raw)::INTEGER,

    NULLIF(NULLIF(TRIM(workclass_raw), '?'), ''),

    TRIM(fnlwgt_raw)::INTEGER,

    TRIM(education_raw),

    TRIM(education_num_raw)::INTEGER,

    TRIM(marital_status_raw),

    NULLIF(NULLIF(TRIM(occupation_raw), '?'), ''),

    TRIM(relationship_raw),

    TRIM(race_raw),

    TRIM(sex_raw),

    TRIM(capital_gain_raw)::INTEGER,

    TRIM(capital_loss_raw)::INTEGER,

    TRIM(hours_per_week_raw)::INTEGER,

    NULLIF(NULLIF(TRIM(native_country_raw), '?'), ''),

    CASE
        WHEN TRIM(income_raw) = '>50K' THEN TRUE
        WHEN TRIM(income_raw) = '<=50K' THEN FALSE
        ELSE NULL
    END

FROM hw3_adult_staging;
"""

with engine.begin() as conn:
    conn.execute(text(insert_clean_sql))

print("Clean data inserted successfully.")

Clean data inserted successfully.


In [9]:
# Compare staging and clean row counts
df = pd.read_sql(text("""
    SELECT 'staging' AS layer, COUNT(*) AS n_rows
    FROM hw3_adult_staging

    UNION ALL

    SELECT 'clean' AS layer, COUNT(*) AS n_rows
    FROM hw3_adult_clean
"""), engine)

display(df)

,layer,n_rows
0,staging,32561
1,clean,32561


In [10]:
# Preview cleaned records
df = pd.read_sql(text("""
    SELECT
        record_id,
        age,
        workclass,
        education,
        occupation,
        hours_per_week,
        native_country,
        income
    FROM hw3_adult_clean
    ORDER BY record_id
    LIMIT 10
"""), engine)

display(df)

,record_id,age,workclass,education,occupation,hours_per_week,native_country,income
0,1,39,State-gov,Bachelors,Adm-clerical,40,United-States,False
1,2,50,Self-emp-not-inc,Bachelors,Exec-managerial,13,United-States,False
2,3,38,Private,HS-grad,Handlers-cleaners,40,United-States,False
3,4,53,Private,11th,Handlers-cleaners,40,United-States,False
4,5,28,Private,Bachelors,Prof-specialty,40,Cuba,False
5,6,37,Private,Masters,Exec-managerial,40,United-States,False
6,7,49,Private,9th,Other-service,16,Jamaica,False
7,8,52,Self-emp-not-inc,HS-grad,Exec-managerial,45,United-States,True
8,9,31,Private,Masters,Prof-specialty,50,United-States,True
9,10,42,Private,Bachelors,Exec-managerial,40,United-States,True


In [11]:
# Check PostgreSQL data types
df = pd.read_sql(text("""
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_name = 'hw3_adult_clean'
      AND table_schema = 'public'
    ORDER BY ordinal_position
"""), engine)

display(df)

,column_name,data_type,is_nullable
0,record_id,bigint,NO
1,age,integer,NO
2,workclass,text,YES
3,fnlwgt,integer,NO
4,education,text,NO
5,education_num,integer,NO
6,marital_status,text,NO
7,occupation,text,YES
8,relationship,text,NO
9,race,text,NO


### Результат

Було створено таблицю `hw3_adult_clean` і перенесено в неї дані з `hw3_adult_staging`.

Під час перенесення числові значення були перетворені на `INTEGER`, текстові поля очищені від зайвих пробілів, а символи `?` у колонках `workclass`, `occupation` і `native_country` замінені на `NULL`.

Колонку `income` було перетворено на `BOOLEAN`, оскільки вона має два значення: дохід більше 50 000 доларів або ні.

Також додано первинний ключ `record_id` і кілька `CHECK`-обмежень для перевірки числових значень.

У результаті кількість записів у двох таблицях однакова - 32 561. Жоден рядок під час очищення не був видалений. Тепер дані готові для перевірки якості та подальшого аналізу.

## 3. Перевірка якості даних

### 3.1. Перевірка кількості рядків

Спочатку перевіримо, чи всі записи були перенесені з початкової таблиці в очищену. Для цього порівняємо кількість рядків у `hw3_adult_staging` та `hw3_adult_clean`.

In [12]:
# Compare row counts between staging and clean tables
df = pd.read_sql(text("""
    SELECT 'staging' AS layer, COUNT(*) AS n_rows
    FROM hw3_adult_staging

    UNION ALL

    SELECT 'clean' AS layer, COUNT(*) AS n_rows
    FROM hw3_adult_clean
"""), engine)

display(df)

,layer,n_rows
0,staging,32561
1,clean,32561


**Висновок:** В обох таблицях по 32 561 запису. Це означає, що під час очищення жоден рядок не був видалений. Усі записи з початкового файлу були перенесені до фінальної таблиці.

### 3.2. Перевірка дублікатів

У датасеті немає унікального ідентифікатора людини, тому перевіримо дублікати за всіма основними колонками, крім `record_id`.

Якщо два рядки мають однакові значення в усіх колонках, вони будуть вважатися потенційними дублікатами. Але це не обов'язково помилка, оскільки різні люди можуть мати однакові характеристики.

In [13]:
# Check exact duplicates based on all original attributes
df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (
            age,
            workclass,
            fnlwgt,
            education,
            education_num,
            marital_status,
            occupation,
            relationship,
            race,
            sex,
            capital_gain,
            capital_loss,
            hours_per_week,
            native_country,
            income
        )) AS distinct_records,
        COUNT(*) - COUNT(DISTINCT (
            age,
            workclass,
            fnlwgt,
            education,
            education_num,
            marital_status,
            occupation,
            relationship,
            race,
            sex,
            capital_gain,
            capital_loss,
            hours_per_week,
            native_country,
            income
        )) AS duplicate_rows
    FROM hw3_adult_clean
"""), engine)

display(df)

,total_rows,distinct_records,duplicate_rows
0,32561,32537,24


In [14]:
# Find repeated records
df = pd.read_sql(text("""
    SELECT
        age,
        workclass,
        education,
        occupation,
        hours_per_week,
        income,
        COUNT(*) AS occurrences
    FROM hw3_adult_clean
    GROUP BY
        age, workclass, fnlwgt, education, education_num,
        marital_status, occupation, relationship, race, sex,
        capital_gain, capital_loss, hours_per_week,
        native_country, income
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
    LIMIT 10
"""), engine)

display(df)

,age,workclass,education,occupation,hours_per_week,income,occurrences
0,25,Private,1st-4th,Priv-house-serv,40,False,3
1,19,Private,HS-grad,Farming-fishing,40,False,2
2,35,Private,HS-grad,Other-service,40,False,2
3,49,Private,7th-8th,Craft-repair,40,False,2
4,90,Private,Some-college,Other-service,35,False,2
5,19,Private,Some-college,Exec-managerial,30,False,2
6,44,Private,Bachelors,Prof-specialty,45,False,2
7,39,Private,HS-grad,Craft-repair,40,False,2
8,30,Private,HS-grad,Other-service,40,False,2
9,42,Private,Some-college,Prof-specialty,40,True,2


**Висновок:** Перевірка показала, що в датасеті є **24 потенційних дублікати**. Це записи, у яких збігаються всі основні атрибути.

Оскільки датасет не містить унікального ID людини, не можна точно сказати, чи це помилкові дублікати. Тому поки залишимо ці записи без змін. Перед видаленням потрібно було б додатково перевірити їх походження.

### 3.3. Перевірка пропущених значень

У початковому датасеті деякі значення були позначені символом `?`. Під час очищення замінимо їх на `NULL`.

Тепер перевіримо, скільки пропусків залишилося в різних колонках і яку частку від загальної кількості записів вони складають.

In [15]:
# Calculate NULL counts and percentages for important columns
df = pd.read_sql(text("""
    SELECT
        column_name,
        missing_count,
        total_rows,
        ROUND(
            missing_count * 100.0 / NULLIF(total_rows, 0),
            2
        ) AS null_percentage
    FROM (
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) FILTER (WHERE workclass IS NULL) AS workclass,
            COUNT(*) FILTER (WHERE occupation IS NULL) AS occupation,
            COUNT(*) FILTER (WHERE native_country IS NULL) AS native_country,
            COUNT(*) FILTER (WHERE age IS NULL) AS age,
            COUNT(*) FILTER (WHERE education IS NULL) AS education,
            COUNT(*) FILTER (WHERE hours_per_week IS NULL) AS hours_per_week,
            COUNT(*) FILTER (WHERE capital_gain IS NULL) AS capital_gain,
            COUNT(*) FILTER (WHERE income IS NULL) AS income
        FROM hw3_adult_clean
    ) AS counts
    CROSS JOIN LATERAL (
        VALUES
            ('workclass', workclass),
            ('occupation', occupation),
            ('native_country', native_country),
            ('age', age),
            ('education', education),
            ('hours_per_week', hours_per_week),
            ('capital_gain', capital_gain),
            ('income', income)
    ) AS missing_values(column_name, missing_count)
    ORDER BY null_percentage DESC, column_name
"""), engine)

display(df)

,column_name,missing_count,total_rows,null_percentage
0,occupation,1843,32561,5.66
1,workclass,1836,32561,5.64
2,native_country,583,32561,1.79
3,age,0,32561,0.00
4,capital_gain,0,32561,0.00
5,education,0,32561,0.00
6,hours_per_week,0,32561,0.00
7,income,0,32561,0.00


In [16]:
# Check special missing-value markers in staging data
df = pd.read_sql(text("""
    SELECT
        column_name,
        COUNT(*) FILTER (WHERE raw_value = '?') AS question_marks,
        COUNT(*) FILTER (WHERE raw_value = '') AS empty_strings,
        COUNT(*) FILTER (WHERE LOWER(raw_value) = 'unknown') AS unknown_values
    FROM (
        SELECT
            'workclass' AS column_name,
            TRIM(workclass_raw) AS raw_value
        FROM hw3_adult_staging

        UNION ALL

        SELECT
            'occupation',
            TRIM(occupation_raw)
        FROM hw3_adult_staging

        UNION ALL

        SELECT
            'native_country',
            TRIM(native_country_raw)
        FROM hw3_adult_staging
    ) AS raw_data
    GROUP BY column_name
    ORDER BY column_name
"""), engine)

display(df)

,column_name,question_marks,empty_strings,unknown_values
0,native_country,583,0,0
1,occupation,1843,0,0
2,workclass,1836,0,0


In [17]:
# Compare missing markers before and after cleaning
df = pd.read_sql(text("""
    SELECT
        'workclass' AS column_name,
        (SELECT COUNT(*)
         FROM hw3_adult_staging
         WHERE TRIM(workclass_raw) = '?') AS raw_missing,
        (SELECT COUNT(*)
         FROM hw3_adult_clean
         WHERE workclass IS NULL) AS clean_nulls

    UNION ALL

    SELECT
        'occupation',
        (SELECT COUNT(*)
         FROM hw3_adult_staging
         WHERE TRIM(occupation_raw) = '?'),
        (SELECT COUNT(*)
         FROM hw3_adult_clean
         WHERE occupation IS NULL)

    UNION ALL

    SELECT
        'native_country',
        (SELECT COUNT(*)
         FROM hw3_adult_staging
         WHERE TRIM(native_country_raw) = '?'),
        (SELECT COUNT(*)
         FROM hw3_adult_clean
         WHERE native_country IS NULL)
"""), engine)

display(df)

,column_name,raw_missing,clean_nulls
0,workclass,1836,1836
1,occupation,1843,1843
2,native_country,583,583


### Висновки щодо якості даних

Під час перевірки ми порівняли кількість записів у двох таблицях, перевірили дублікати та пропущені значення.

У staging та clean таблицях залишилося по 32 561 запису, тобто під час очищення дані не були втрачені.

Найбільше пропусків виявилося в колонках `occupation` (5.66%) та `workclass` (5.64%). У колонці `native_country` пропущено 1.79% значень. В інших перевірених колонках пропусків немає.

У початковому файлі пропуски були позначені символом `?`. Після очищення вони стали SQL `NULL`, що я додатково перевірила окремим запитом.

Також перевірили записи на дублікати. Оскільки в датасеті немає унікального ідентифікатора людини, навіть повністю однакові рядки не обов'язково є помилкою. Тому видаляти їх без додаткової інформації ми не стали.

Загалом дані можна використовувати для подальшого аналізу, але перед навчанням ML-моделі потрібно буде вирішити, як обробляти пропущені категоріальні значення.

## 4. Розподіли та потенційні аномалії

### 4.1. Розподіл доходів

Було проаналізовано розподіл цільової змінної `income`, яка показує, чи перевищує річний дохід людини 50 000 доларів.

Це дозволяє оцінити баланс класів, що важливо для подальшого навчання ML-моделі.

In [18]:
# Analyze the distribution of income categories
df = pd.read_sql(text("""
    SELECT
        CASE
            WHEN income = TRUE THEN '>50K'
            ELSE '<=50K'
        END AS income_category,
        COUNT(*) AS n_rows,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS pct_rows
    FROM hw3_adult_clean
    GROUP BY income
    ORDER BY n_rows DESC
    LIMIT 10
"""), engine)

display(df)

,income_category,n_rows,pct_rows
0,<=50K,24720,75.92
1,>50K,7841,24.08


**Висновок:** Більшість записів (75.92%) належить до категорії доходу до 50 000 доларів. Лише 24.08% мають дохід понад 50 000.

Отже, класи в датасеті незбалансовані. Це потрібно враховувати під час навчання ML-моделі, оскільки звичайна accuracy може показувати досить хороший результат, навіть якщо модель погано визначає менш поширений клас.

### 4.2. Статистика робочих годин

Для аналізу числових даних було обрано колонку `hours_per_week`. Вона містить кількість робочих годин на тиждень.

Було розраховано мінімальне, максимальне та середнє значення, а також стандартне відхилення, щоб оцінити розкид даних.

In [19]:
# Calculate basic statistics for weekly working hours
df = pd.read_sql(text("""
    SELECT
        COUNT(hours_per_week) AS n_non_null,
        MIN(hours_per_week) AS min_value,
        MAX(hours_per_week) AS max_value,
        ROUND(AVG(hours_per_week)::NUMERIC, 2) AS avg_value,
        ROUND(STDDEV_SAMP(hours_per_week)::NUMERIC, 2) AS sd_value
    FROM hw3_adult_clean
"""), engine)

display(df)

,n_non_null,min_value,max_value,avg_value,sd_value
0,32561,1,99,40.44,12.35


**Висновок:** У середньому люди в датасеті працюють близько 40.44 години на тиждень. Мінімальне значення становить 1 годину, а максимальне - 99 годин.

Стандартне відхилення становить приблизно 12.35 години. Це показує, що кількість робочих годин досить сильно відрізняється між записами.

Особливо високі значення варто перевірити окремо, оскільки вони можуть бути як реальними випадками, так і неточностями у даних.

### 4.3. Перевірка аномальних значень

Для пошуку потенційних аномалій було обрано записи, у яких кількість робочих годин перевищує 80 на тиждень.

Такі значення можуть бути реальними, але вони значно вищі за звичайний робочий графік. Тому їх варто розглянути окремо.

In [20]:
# Find records with unusually high weekly working hours
df = pd.read_sql(text("""
    SELECT
        record_id,
        age,
        workclass,
        occupation,
        hours_per_week,
        income
    FROM hw3_adult_clean
    WHERE hours_per_week > 80
    ORDER BY hours_per_week DESC, record_id
    LIMIT 20
"""), engine)

display(df)

,record_id,age,workclass,occupation,hours_per_week,income
0,937,37,Private,Craft-repair,99,True
1,1174,25,Private,Farming-fishing,99,True
2,1889,55,Self-emp-not-inc,Farming-fishing,99,True
3,3580,37,Self-emp-inc,Prof-specialty,99,True
4,4088,50,None,None,99,False
5,4310,35,Self-emp-not-inc,Farming-fishing,99,False
6,4443,43,Self-emp-inc,Prof-specialty,99,True
7,5378,43,Private,Craft-repair,99,False
8,5434,44,Private,Prof-specialty,99,False
9,6620,27,Private,Transport-moving,99,False


In [21]:
# Count records with more than 80 working hours per week
df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE hours_per_week > 80
        ) AS outlier_rows,
        ROUND(
            COUNT(*) FILTER (
                WHERE hours_per_week > 80
            ) * 100.0 / NULLIF(COUNT(*), 0),
            2
        ) AS outlier_percentage
    FROM hw3_adult_clean
"""), engine)

display(df)

,total_rows,outlier_rows,outlier_percentage
0,32561,208,0.64


**Висновок:** Було перевірено записи, у яких кількість робочих годин перевищує 80 на тиждень.

За результатами запиту знайдено **208 таких записів**, що становить **0.64%** від усього датасету.

Ці значення виглядають незвично, але не обов'язково є помилками. Наприклад, деякі люди можуть працювати на кількох роботах або мати дуже довгий робочий тиждень.

Тому такі записи поки не видалялися. Для подальшого аналізу можна додати окрему ознаку, яка позначатиме випадки з понад 80 робочими годинами.

## 5. DQL/EDA-запити

У цьому розділі було виконано 10 SQL-запитів для аналізу датасету Adult.

Основна мета - дослідити, як дохід пов'язаний із віком, освітою, зайнятістю та іншими характеристиками людей.

Для аналізу використовуються різні SQL-конструкції, зокрема `WHERE`, `GROUP BY`, `DISTINCT`, `LIKE`, `COALESCE` та інші.

In [22]:
# Query 1: Income distribution
q1 = pd.read_sql(text("""
    SELECT
        income,
        COUNT(*) AS people_count,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS percentage
    FROM hw3_adult_clean
    GROUP BY income
    ORDER BY people_count DESC
"""), engine)

display(q1)

low = q1.loc[q1["income"] == False].iloc[0]
high = q1.loc[q1["income"] == True].iloc[0]

print(
    f"Інтерпретація: Дохід до 50 000 доларів мають "
    f"{low['people_count']} людей ({low['percentage']}%). "
    f"Понад 50 000 доларів заробляють "
    f"{high['people_count']} людей ({high['percentage']}%). "
    "Отже, у датасеті значно більше людей з нижчим доходом."
)

,income,people_count,percentage
0,False,24720,75.92
1,True,7841,24.08


Інтерпретація: Дохід до 50 000 доларів мають 24720 людей (75.92%). Понад 50 000 доларів заробляють 7841 людей (24.08%). Отже, у датасеті значно більше людей з нижчим доходом.


In [23]:
# Query 2: Income distribution by education
q2 = pd.read_sql(text("""
    SELECT
        education,
        COUNT(*) AS total_people,
        COUNT(*) FILTER (WHERE income = TRUE) AS high_income,
        ROUND(
            COUNT(*) FILTER (WHERE income = TRUE) * 100.0
            / COUNT(*),
            2
        ) AS high_income_pct
    FROM hw3_adult_clean
    GROUP BY education
    HAVING COUNT(*) >= 100
    ORDER BY high_income_pct DESC
    LIMIT 5
"""), engine)

display(q2)

top = q2.iloc[0]

print(
    f"Інтерпретація: Серед рівнів освіти з не менше ніж "
    f"100 записами найбільша частка високого доходу "
    f"спостерігається для {top['education']} — "
    f"{top['high_income_pct']}%. "
    "Це показує, що рівень освіти може бути пов'язаний "
    "із доходом. Але сама освіта не обов'язково є "
    "причиною вищого доходу."
)

,education,total_people,high_income,high_income_pct
0,Doctorate,413,306,74.09
1,Prof-school,576,423,73.44
2,Masters,1723,959,55.66
3,Bachelors,5355,2221,41.48
4,Assoc-voc,1382,361,26.12


Інтерпретація: Серед рівнів освіти з не менше ніж 100 записами найбільша частка високого доходу спостерігається для Doctorate — 74.09%. Це показує, що рівень освіти може бути пов'язаний із доходом. Але сама освіта не обов'язково є причиною вищого доходу.


In [24]:
# Query 3: Most common employment types
q3 = pd.read_sql(text("""
    SELECT
        COALESCE(workclass, 'Unknown') AS workclass_name,
        COUNT(*) AS people_count
    FROM hw3_adult_clean
    GROUP BY COALESCE(workclass, 'Unknown')
    ORDER BY people_count DESC
    LIMIT 5
"""), engine)

display(q3)

top = q3.iloc[0]

print(
    f"Інтерпретація: Найпоширеніший тип зайнятості — "
    f"{top['workclass_name']}, до нього належать "
    f"{top['people_count']} записів. "
    "Для пропущених значень використано назву Unknown. "
    "Це дозволяє врахувати такі записи, не видаляючи їх."
)

,workclass_name,people_count
0,Private,22696
1,Self-emp-not-inc,2541
2,Local-gov,2093
3,Unknown,1836
4,State-gov,1298


Інтерпретація: Найпоширеніший тип зайнятості — Private, до нього належать 22696 записів. Для пропущених значень використано назву Unknown. Це дозволяє врахувати такі записи, не видаляючи їх.


In [25]:
# Query 4: Filter people by age, hours, education, and income
q4 = pd.read_sql(text("""
    SELECT
        age,
        education,
        hours_per_week,
        income
    FROM hw3_adult_clean
    WHERE (
        (age BETWEEN 25 AND 40 AND hours_per_week >= 40)
        OR
        (education IN ('Bachelors') AND NOT income)
    )
    ORDER BY age, hours_per_week DESC
    LIMIT 20
"""), engine)

display(q4)

print(
    f"Інтерпретація: Запит показав перші {len(q4)} записів, "
    "які відповідають хоча б одній із двох умов. "
    "Використання AND, OR та дужок дозволяє правильно "
    "об'єднати кілька критеріїв відбору. "
    "Результат обмежено 20 рядками для перегляду."
)

,age,education,hours_per_week,income
0,19,Bachelors,40,False
1,19,Bachelors,30,False
2,20,Bachelors,30,False
3,21,Bachelors,45,False
4,21,Bachelors,40,False
5,21,Bachelors,40,False
6,21,Bachelors,30,False
7,21,Bachelors,25,False
8,21,Bachelors,20,False
9,21,Bachelors,20,False


Інтерпретація: Запит показав перші 20 записів, які відповідають хоча б одній із двох умов. Використання AND, OR та дужок дозволяє правильно об'єднати кілька критеріїв відбору. Результат обмежено 20 рядками для перегляду.


In [26]:
# Query 5: Find technology-related occupation names
q5 = pd.read_sql(text("""
    SELECT
        occupation,
        age,
        education,
        hours_per_week,
        income
    FROM hw3_adult_clean
    WHERE occupation ILIKE '%tech%'
    ORDER BY occupation, age
    LIMIT 20
"""), engine)

display(q5)

categories = ", ".join(q5["occupation"].dropna().unique())

print(
    f"Інтерпретація: За умовою пошуку знайдено "
    f"такі професії серед перших 20 записів: {categories}. "
    "Оператор ILIKE дозволяє шукати текст без урахування "
    "великих і малих літер. Це зручно для пошуку "
    "певних категорій у текстових колонках."
)

,occupation,age,education,hours_per_week,income
0,Tech-support,17,10th,10,False
1,Tech-support,18,HS-grad,10,False
2,Tech-support,18,Some-college,20,False
3,Tech-support,19,HS-grad,40,False
4,Tech-support,19,Some-college,14,False
5,Tech-support,19,Some-college,25,False
6,Tech-support,19,Some-college,16,False
7,Tech-support,19,HS-grad,40,False
8,Tech-support,19,Some-college,30,False
9,Tech-support,19,Some-college,40,False


Інтерпретація: За умовою пошуку знайдено такі професії серед перших 20 записів: Tech-support. Оператор ILIKE дозволяє шукати текст без урахування великих і малих літер. Це зручно для пошуку певних категорій у текстових колонках.


In [27]:
# Query 6: Find unique countries
q6 = pd.read_sql(text("""
    SELECT DISTINCT
        native_country
    FROM hw3_adult_clean
    WHERE native_country IS NOT NULL
    ORDER BY native_country
    LIMIT 20
"""), engine)

display(q6)

print(
    f"Інтерпретація: Запит показав {len(q6)} різних "
    "країн походження з датасету. "
    "DISTINCT дозволяє прибрати повторення, "
    "а IS NOT NULL виключає пропущені значення. "
    "Для зручності перегляду список обмежено 20 країнами."
)

,native_country
0,Cambodia
1,Canada
2,China
3,Columbia
4,Cuba
5,Dominican-Republic
6,Ecuador
7,El-Salvador
8,England
9,France


Інтерпретація: Запит показав 20 різних країн походження з датасету. DISTINCT дозволяє прибрати повторення, а IS NOT NULL виключає пропущені значення. Для зручності перегляду список обмежено 20 країнами.


In [28]:
# Query 7: Income distribution for missing occupations
q7 = pd.read_sql(text("""
    SELECT
        income,
        COUNT(*) AS people_count
    FROM hw3_adult_clean
    WHERE occupation IS NULL
    GROUP BY income
    ORDER BY income
"""), engine)

display(q7)

missing_total = q7["people_count"].sum()
high_total = q7.loc[q7["income"] == True, "people_count"].sum()

print(
    f"Інтерпретація: У {missing_total} записах "
    f"відсутня інформація про професію. "
    f"Серед них {high_total} людей мають дохід понад 50 000 доларів. "
    "Ці пропуски варто враховувати під час аналізу, "
    "оскільки професія може бути важливою ознакою для ML-моделі."
)

,income,people_count
0,False,1652
1,True,191


Інтерпретація: У 1843 записах відсутня інформація про професію. Серед них 191 людей мають дохід понад 50 000 доларів. Ці пропуски варто враховувати під час аналізу, оскільки професія може бути важливою ознакою для ML-моделі.


In [29]:
# Query 8: Compare occupations, including NULL values
q8 = pd.read_sql(text("""
    SELECT
        COUNT(*) FILTER (
            WHERE occupation = 'Prof-specialty'
        ) AS prof_specialty_count,

        COUNT(*) FILTER (
            WHERE occupation IS DISTINCT FROM 'Prof-specialty'
        ) AS other_or_unknown_count,

        COUNT(*) FILTER (
            WHERE occupation IS NULL
        ) AS unknown_count
    FROM hw3_adult_clean
"""), engine)

display(q8)

r = q8.iloc[0]

print(
    f"Інтерпретація: Професію Prof-specialty мають "
    f"{r['prof_specialty_count']} записів. "
    f"Інші або невідомі професії мають "
    f"{r['other_or_unknown_count']} записів, із них "
    f"{r['unknown_count']} — NULL. "
    "IS DISTINCT FROM враховує NULL як значення, "
    "відмінне від Prof-specialty, на відміну від звичайного <>."
)

,prof_specialty_count,other_or_unknown_count,unknown_count
0,4140,28421,1843


Інтерпретація: Професію Prof-specialty мають 4140 записів. Інші або невідомі професії мають 28421 записів, із них 1843 — NULL. IS DISTINCT FROM враховує NULL як значення, відмінне від Prof-specialty, на відміну від звичайного <>.


In [30]:
# Query 9: Income by working-hours category
q9 = pd.read_sql(text("""
    SELECT
        CASE
            WHEN hours_per_week < 35 THEN 'Less than 35'
            WHEN hours_per_week BETWEEN 35 AND 40 THEN '35-40'
            WHEN hours_per_week BETWEEN 41 AND 50 THEN '41-50'
            ELSE 'More than 50'
        END AS hours_group,

        COUNT(*) AS total_people,

        ROUND(
            COUNT(*) FILTER (WHERE income = TRUE) * 100.0
            / COUNT(*),
            2
        ) AS high_income_pct

    FROM hw3_adult_clean
    GROUP BY hours_group
    ORDER BY high_income_pct DESC
"""), engine)

display(q9)

top = q9.iloc[0]
bottom = q9.iloc[-1]

print(
    f"Інтерпретація: Найбільша частка людей із доходом "
    f"понад 50 000 доларів спостерігається в групі "
    f"{top['hours_group']} ({top['high_income_pct']}%). "
    f"Найменша — у групі {bottom['hours_group']} "
    f"({bottom['high_income_pct']}%). "
    "Це показує зв'язок між робочими годинами та доходом, "
    "але не доводить, що саме тривалість роботи визначає дохід."
)

,hours_group,total_people,high_income_pct
0,More than 50,3643,41.28
1,41-50,5938,39.61
2,35-40,17397,20.68
3,Less than 35,5583,6.93


Інтерпретація: Найбільша частка людей із доходом понад 50 000 доларів спостерігається в групі More than 50 (41.28%). Найменша — у групі Less than 35 (6.93%). Це показує зв'язок між робочими годинами та доходом, але не доводить, що саме тривалість роботи визначає дохід.


In [31]:
# Query 10: Compare income groups by capital gains
q10 = pd.read_sql(text("""
    SELECT
        CASE
            WHEN capital_gain > 0 THEN 'Has capital gain'
            ELSE 'No capital gain'
        END AS capital_gain_group,

        COUNT(*) AS total_people,

        COUNT(*) FILTER (
            WHERE income = TRUE
        ) AS high_income_count,

        ROUND(
            COUNT(*) FILTER (WHERE income = TRUE) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS high_income_pct

    FROM hw3_adult_clean
    GROUP BY (capital_gain > 0)
    ORDER BY high_income_pct DESC
"""), engine)

display(q10)

with_gain = q10.loc[
    q10["capital_gain_group"] == "Has capital gain"
].iloc[0]

without_gain = q10.loc[
    q10["capital_gain_group"] == "No capital gain"
].iloc[0]

print(
    f"Інтерпретація: Серед людей із доходом від капіталу "
    f"частка високого доходу становить "
    f"{with_gain['high_income_pct']}%. "
    f"Серед людей без такого доходу — "
    f"{without_gain['high_income_pct']}%. "
    "Отже, capital_gain може бути корисною ознакою "
    "для прогнозування категорії доходу."
)

,capital_gain_group,total_people,high_income_count,high_income_pct
0,Has capital gain,2712,1677,61.84
1,No capital gain,29849,6164,20.65


Інтерпретація: Серед людей із доходом від капіталу частка високого доходу становить 61.84%. Серед людей без такого доходу — 20.65%. Отже, capital_gain може бути корисною ознакою для прогнозування категорії доходу.


### Висновок

Було виконано 10 SQL-запитів для аналізу даних про доходи, освіту, професії та робочі години.

Результати показують, що в датасеті є значний дисбаланс між двома категоріями доходу. Також було досліджено, як частка людей із високим доходом відрізняється залежно від освіти, робочих годин та наявності доходу від капіталу.

Окремо було перевірено пропущені значення у професіях і використано різні SQL-оператори для фільтрації та групування даних.

Такий аналіз допомагає краще зрозуміти датасет перед використанням у машинному навчанні.

## 6. Reflection

**1. Які проблеми якості даних було виявлено?**

Під час перевірки датасету Adult було знайдено пропущені значення в колонках `workclass`, `occupation` та `native_country`. У початковому файлі вони були позначені символом `?`, тому під час очищення їх замінили на `NULL`. Також було перевірено потенційні дублікати та незвично великі значення робочих годин. Дублікати вирішено не видаляти, оскільки немає можливості точно визначити, чи належать однакові записи різним людям.

**2. Які колонки можна використати для ML?**

Для прогнозування доходу можна використати `age`, `education`, `occupation`, `workclass`, `hours_per_week`, `capital_gain` та `capital_loss`. Ці характеристики можуть бути пов'язані з рівнем доходу. Колонку `record_id` використовувати не варто, оскільки це просто технічний ідентифікатор. Також не всі категоріальні ознаки обов'язково потрібні для моделі.

**3. Чи є ризик data leakage?**

Колонка `income` є цільовою змінною, тому її не можна використовувати як вхідну ознаку. Для інших полів ризик залежить від того, коли саме робиться прогноз. Наприклад, `capital_gain` можна використовувати тільки тоді, коли це значення вже відоме на момент прогнозування.

**4. Як можна покращити процес завантаження даних?**

Було б корисно автоматично перевіряти типи даних, кількість пропусків, дублікати та допустимі діапазони значень. Також варто використовувати однакові позначення для пропущених значень, а не різні спеціальні символи.

**5. Яка ML-задача відповідає датасету?**

Для Adult найбільше підходить бінарна класифікація. Модель повинна визначати, чи перевищує річний дохід людини 50 000 доларів. Оскільки класи незбалансовані, для оцінювання моделі варто використовувати не тільки accuracy, але й precision, recall та F1-score.